# kickmix hello world

A five-minute tour of [kickmix](https://github.com/quantumlib/Qualtran/tree/main/kickmix) through the [kickmix-macos fork](https://github.com/manwithacat/kickmix-macos): build a small reversible circuit, simulate a batch of shots, read registers back and look at the operation statistics. Optionally run one of the Kaggle competition's official verifiers.

The fork is an openly AI-assisted variant of upstream kickmix (Google LLC, Apache-2.0); this notebook and the fork's patches were written with an AI coding model (Claude) under the repository owner's direction. The Python API shown here is upstream's.

## Install

If `kickmix` is not importable, install the fork from GitHub (compiles C++; a few minutes) or, faster, a prebuilt wheel from the [releases page](https://github.com/manwithacat/kickmix-macos/releases). In CI the wheel built from the checked-out commit is already installed, so this cell does nothing.

In [1]:
try:
    import kickmix as km
except ImportError:
    %pip install --quiet numpy "git+https://github.com/manwithacat/kickmix-macos.git#subdirectory=kickmix"
    import kickmix as km
import numpy as np
print("kickmix from", km.__file__)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.
kickmix from /usr/local/lib/python3.13/dist-packages/kickmix.cpython-313-x86_64-linux-gnu.so


## Build a circuit

A `CircuitBuilder` allocates quantum registers and appends operations. Here `iadd` adds register `b` into register `a`, modulo 2**4, reversibly.

In [2]:
builder = km.CircuitBuilder()
a = builder.create_quantum_register(4, "a")
b = builder.create_quantum_register(4, "b")
builder.iadd(b, target=a)           # a += b (mod 16)
circuit = builder.finish_circuit()
print(circuit.num_qubits, "qubits")
print(circuit.text_diagram())

10 qubits
q0: -a[0]-----@-------------------------------------------@-----X-
              |                                           |     |
q1: -a[1]-----|-X---@--------------------@-----X-X--------|-----|-
              | |   |                    |     | |        |     |
q2: -a[2]-----|-|---|---X-@-X-X----------|-----|-|--------|-----|-
              | |   |   | | | |          |     | |        |     |
q3: -a[3]-X---|-|---|---X-X-|-|----------|-----|-|--------|-----|-
          |   | |   |   | | | |          |     | |        |     |
q4: -b[0]-|---@-|---|---|-|-|-|----------|-----|-|--------Z**b0-@-
          |   | |   |   | | | |          |     | |
q5: -b[1]-|---|-X---@---|-|-|-|----------Z**b0-@-X----------------
          |   | |   |   | | | |                  |
q6: -b[2]-|---|-|---|---X-@-@-X------------------|----------------
          |   | |   |   |     |                  |
q7: -b[3]-@---|-|---|---|-----|------------------|----------------
              | |   |   |     |      

## Simulate a batch

A `Simulator` runs `batch_size` independent shots at once (256 per machine word). `write_across_shots` loads one bit-plane per qubit; here random 4-bit values. `counting_operations=True` is what the competition verifiers use.

In [3]:
shots = 256
rng = np.random.default_rng(0)
a_in = rng.integers(0, 16, size=shots)
b_in = rng.integers(0, 16, size=shots)
to_bits = lambda values: ((values[None, :] >> np.arange(4)[:, None]) & 1).astype(bool)   # (4, shots), little-endian

sim = km.Simulator(batch_size=shots, debug_print=False, seed=1, counting_operations=True)
sim.write_across_shots(a, to_bits(a_in))
sim.write_across_shots(b, to_bits(b_in))
sim.do(circuit)

## Read the registers

In [4]:
bits = sim.read_across_shots(a)                    # bool array, shape (4, shots)
a_out = (bits.astype(int) * (1 << np.arange(4))[:, None]).sum(axis=0)
b_out = (sim.read_across_shots(b).astype(int) * (1 << np.arange(4))[:, None]).sum(axis=0)
assert (a_out == (a_in + b_in) % 16).all(), "a should hold a + b mod 16"
assert (b_out == b_in).all(), "b is unchanged"
print("first five shots:", [(int(x), int(y), int(z)) for x, y, z in zip(a_in[:5], b_in[:5], a_out[:5])], "(a, b, a+b mod 16)")
print("phase flipped in any shot:", bool(np.any(sim.read_phase_flipped_across_shots())))

first five shots: [(13, 2, 15), (10, 1, 11), (8, 12, 4), (4, 4, 8), (4, 12, 0)] (a, b, a+b mod 16)
phase flipped in any shot: False


## Operation statistics

With counting on, the simulator tallies operations, summed over shots or for one shot. `num_toffoli_like` is the Toffoli count the competition scores.

In [5]:
stats = sim.operation_statistics()                 # summed over all shots
one = sim.operation_statistics(shot_index=0)
print("summed over", shots, "shots:", stats)
print("Toffoli-like operations per shot:", one.num_toffoli_like, "(summed:", stats.num_toffoli_like, ")")
assert stats.num_toffoli_like == shots * one.num_toffoli_like     # unconditional circuit: same in every shot

summed over 256 shots: <km.OperationStatistics num_instructions=6397 num_toffoli_like=768 num_t_like=0>
Toffoli-like operations per shot: 3 (summed: 768 )


## Optional: an official verifier

The competition's verifiers are not part of this repository. If you have the competition files from Kaggle, set `OFFICIAL_DIR` to the directory that holds `verifiers/` and `sample_submission/`; the next cell then verifies the sample chemistry family with this build of kickmix (about half a minute). Left empty, it is skipped.

In [6]:
OFFICIAL_DIR = ""     # e.g. "/kaggle/input/google-verifiable-quantum-circuit-compilation" or a local copy

import sys
from pathlib import Path
if OFFICIAL_DIR:
    sys.path.insert(0, str(Path(OFFICIAL_DIR) / "verifiers"))
    from dfthcsos.verify import verify as verify_chemistry
    result = verify_chemistry(Path(OFFICIAL_DIR) / "sample_submission" / "dfthcsos")
    print({k: result[k] for k in ("status", "score", "toffoli_count", "num_qubits")})
else:
    print("OFFICIAL_DIR not set; skipping")

OFFICIAL_DIR not set; skipping
